# Notebook 02 — TrOCR Fine-tuning on IAM

This notebook:
1. Loads the IAM words dataset with train/val/test splits (80/10/10)
2. Defines the PyTorch Dataset + collate function for TrOCRProcessor
3. Configures and runs HuggingFace Trainer with Seq2SeqTrainingArguments
4. Monitors val CER per epoch with early stopping (patience=3)
5. Saves best checkpoint to models/trocr_finetuned/
6. Evaluates on test set and logs final CER/WER vs baseline

**Requires:** CUDA GPU (8GB+ VRAM recommended), IAM dataset downloaded to data/iam/

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import jiwer
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    print('WARNING: Training on CPU will be extremely slow. Use a CUDA GPU.')

In [ ]:
# ── IAM Dataset ────────────────────────────────────────────────────────────────

class IAMWordsDataset(Dataset):
    """IAM words split dataset for TrOCR fine-tuning."""
    
    def __init__(self, samples, processor, max_target_length=64):
        self.samples = samples
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s['path']).convert('RGB')
        pixel_values = self.processor(images=image, return_tensors='pt').pixel_values.squeeze()
        labels = self.processor.tokenizer(
            text=s['label'],
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze()
        # Replace PAD token id with -100 so loss ignores padding
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}

In [ ]:
# ── Load samples and split ─────────────────────────────────────────────────────
IAM_DIR = Path('../data/iam')

# Load all samples (same as notebook 00)
samples = []  # ... (load from words.txt)

# 80/10/10 split
n = len(samples)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val
train_s, val_s, test_s = samples[:n_train], samples[n_train:n_train+n_val], samples[n_train+n_val:]

print(f'Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}')

In [ ]:
# ── Model & Processor ──────────────────────────────────────────────────────────
MODEL_NAME = 'microsoft/trocr-base-handwritten'
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Required configuration
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

train_ds = IAMWordsDataset(train_s, processor)
val_ds   = IAMWordsDataset(val_s,   processor)
test_ds  = IAMWordsDataset(test_s,  processor)

print(f'Model loaded. Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── CER Metric ────────────────────────────────────────────────────────────────

def compute_metrics(pred):
    label_ids = pred.label_ids
    pred_ids  = pred.predictions
    
    # Replace -100 in labels (padding)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    
    cer = jiwer.cer(label_str, pred_str)
    wer = jiwer.wer(label_str, pred_str)
    return {'cer': cer, 'wer': wer}

In [ ]:
# ── Training Arguments ────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir='../models/trocr_finetuned',
    predict_with_generate=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    learning_rate=5e-5,
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=3,
    report_to='none',  # set to 'wandb' to enable W&B logging
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Trainer configured. Run trainer.train() to start fine-tuning.')

In [ ]:
# ── TRAINING (uncomment to run) ────────────────────────────────────────────────
# trainer.train()

# ── EVALUATION ────────────────────────────────────────────────────────────────
# test_results = trainer.predict(test_ds)
# cer = test_results.metrics['test_cer']
# wer = test_results.metrics['test_wer']
# print(f'Test CER: {cer*100:.2f}%  |  Test WER: {wer*100:.2f}%')
# print('Target: CER <= 8%')
print('Uncomment training and evaluation cells to run.')